In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

claimants = spark.table("scottish_housing_ws.silver.dwp_claimant_count")
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
hpi_councils = (
    hpi
    .filter(F.col("area_code").startswith("S12"))
    .select("report_date", "year_month", "region_name", "area_code", "average_price", "sales_volume")
)

In [0]:
gold = (
    hpi_councils
    .join(
        claimants.select("year_month", "council_area_code", "council_name", "claimant_count"),
        on=[
            hpi_councils.year_month == claimants.year_month,
            hpi_councils.area_code == claimants.council_area_code
        ],
        how="inner"
    )
    .select(
        "report_date",
        hpi_councils.year_month,
        "region_name",
        "area_code",
        "average_price",
        "sales_volume",
        "claimant_count"
    )
    .orderBy("area_code", "report_date")
)

In [0]:
print(f"Row count: {gold.count()}")
gold.filter(F.col("area_code") == "S12000036").orderBy("report_date").show(5)
gold.filter(F.col("area_code") == "S12000036").orderBy(F.col("report_date").desc()).show(5)


In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.claimant_count_vs_transaction_volume")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_claimant_count_vs_transaction_volume/")
)

print("Gold 05 complete.")

In [0]:
# Sources: silver.dwp_claimant_count, silver.uk_hpi
# Grain: monthly, by council area (32 Scottish council areas)
# Join keys: year_month + area_code
#
# Claimant count is a proxy for local unemployment -- people claiming
# unemployment-related benefits. Higher claimant count = weaker local
# labour market = expect lower transaction volumes and price growth.
#
# Coverage overlap: DWP claimant count Apr 2016 to Mar 2026
#                   UK HPI Jan 1968 to Feb 2026
# Inner join will constrain to Apr 2016 to Feb 2026.